# 🎯 [DAT10_M07] 동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우 설계
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M07 |
| **미션 주제** | 동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우 설계 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **동기화 충돌이 발생했을 때 시스템이 값을 자동 확정하지 않고 군수관이 수동 병합하며, 영향이 큰 건은 지휘관이 승인하는 HITL 워크플로우를 구현해 검토 큐 대기시간과 처리 로깅 성공률 100%를 확인한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **충돌 12건(실사 수량·최소 보유 기준을 두 보급소가 다르게 수정)을 시스템 제안 - 보급담당 상신 - 군수관 검토와 수동 병합 - 확정(차이 25% 이상이면 지휘관 승인) 순서로 처리하고, 병합 전후 값과 사유·근거를 기록하며 반려 사유를 필수로 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/dat10_approval_workflow.py 및 docs/dat10_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""동기화 충돌 건을 시스템 제안 -> 보급담당 상신 -> 군수관 수동 병합 -> (영향이 큰 건은 지휘관 승인) 순서의 HITL 워크플로우로 처리하되, 자동 확정 금지·병합 전후 값과 사유 기록·반려 사유 필수화·권한 우회 차단·해시 체인 감사 로그·지휘관 피드백 입력 검증을 구현하고 위험도 우선 큐(대기 노화 가산)와 선입선출의 대기시간을 비교한다."""
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 야전 보급소·업체·품목 자료 생성기 (실제 부대·업체·품목·재고 정보 없음)
# 가상 보급소 5곳이 품목 12종의 입고·불출·조정 트랜잭션을 오프라인으로 기록하고 나중에 동기화한다.
# 공급망 재난 징후(업체 납기 지연, 품목 재고 급감)는 동기화된 자료를 마이닝해 식별한다.
# ---------------------------------------------------------------------------
NODES = ["보급소A", "보급소B", "보급소C", "보급소D", "보급소E"]
SUPPLIERS = ["가상업체1", "가상업체2", "가상업체3", "가상업체4", "가상업체5", "가상업체6"]
CATS = ["일반보급", "정비부품", "의무소모"]
ITEMS = [{"item": f"I{i + 1:02d}", "cat": CATS[i % 3], "supplier": SUPPLIERS[i % 6], "lead_days": 3 + (i * 5) % 12, "use": 4 + (i * 7) % 11, "min_level": 30 + 10 * (i % 5)} for i in range(12)]
ITEM_IDS = [x["item"] for x in ITEMS]
CTXS = ["평시", "야간", "통신불량"]


def make_txns(seed=7, days=7, per_node_day=60, nodes=None):
    """노드별 오프라인 트랜잭션 목록. id는 노드-일련번호(멱등 키), t는 시각(시간 단위), ctx는 입력 상황."""
    rng = np.random.default_rng(seed)
    out = []
    for n in (nodes or NODES):
        seq = 0
        for k in range(days * per_node_day):
            seq += 1
            it = ITEMS[int(rng.integers(len(ITEMS)))]
            t = float(rng.uniform(0, 24 * days))
            ctx = "야간" if (t % 24 >= 20 or t % 24 < 5) else "평시"
            if rng.random() < 0.12:
                ctx = "통신불량"
            r = rng.random()
            if r < 0.22:
                op, q = "in", int(it["use"] * rng.uniform(4, 9))
            elif r < 0.95:
                op, q = "out", int(max(1, rng.normal(it["use"] * 1.5, 3)))
            else:
                op, q = "adjust", int(rng.integers(-5, 6))
            out.append({"id": f"{n}-{seq:05d}", "node": n, "item": it["item"], "op": op, "qty": q, "t": round(t, 2), "ctx": ctx, "supplier": it["supplier"]})
    return out


def stock_of(txns, base=500):
    """품목별 재고 = 기초 재고 + 입고 - 불출 + 조정 (id 중복은 한 번만 반영)."""
    # [TODO : 학생 실습 7] 동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
PERM = {"시스템": {"propose"}, "보급담당": {"submit", "redraft", "view"}, "군수관": {"review", "merge", "confirm", "reject", "view"}, "지휘관": {"approve", "reject", "feedback", "view"}, "감사관": {"view", "audit"}}
USERS = {"sys": "시스템", "kim": "보급담당", "lee": "군수관", "park": "지휘관", "jung": "감사관"}
IMPACT_CUT = 0.25                                                       # 두 값의 차이가 큰 쪽 값의 25% 이상이면 지휘관 승인 필요
MAX_REDRAFT = 2
FB_CATS = {"우선순위", "자원", "절차", "기타"}                          # 지휘관 피드백 입력 폼 분류
SLA_H = 4.0                                                             # 군수관 검토 대기 허용 시간(h)


class Chain:
    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


class Workflow:
    def __init__(self):
        self.audit, self.cases, self.calls = Chain(), {}, 0

    def _deny(self, who, case, action, why):
        self.audit.add(event="DENIED", who=who, case=case, action=action, reason=why); return False, why

    def act(self, who, case, action, reason="", evidence="", value=None, cat=None, cf=None):
        self.calls += 1
        role = USERS.get(who)
        if role is None:
            return self._deny(who, case, action, "미등록 사용자")
        if action in ("delete_log", "edit_log"):
            return self._deny(who, case, action, "감사 로그 삭제·수정 권한은 누구에게도 없음")
        if action not in PERM[role]:
            return self._deny(who, case, action, f"권한 없음({role} 역할에 {action} 권한 없음)")
        if action == "propose":
            a, b = cf
            self.cases[case] = {"state": "제안", "submitter": None, "version": 1, "a": a, "b": b, "merged": None, "high": abs(a - b) / max(abs(a), abs(b), 1) >= IMPACT_CUT, "feedback": []}
            self.audit.add(event="PROPOSE", who=who, case=case, a=a, b=b, reason=reason); return True, "제안 등록(자동 확정 없음)"
        c = self.cases.get(case)
        if c is None:
            return self._deny(who, case, action, "존재하지 않는 건")
        if c["state"] in ("확정", "종결(반려)"):
            return self._deny(who, case, action, f"이미 종결된 건({c['state']})")
        if not reason.strip() or not evidence.strip():
            return self._deny(who, case, action, "사유·근거 입력 필수(반려·병합 사유 포함)")
        if action == "feedback":
            if cat not in FB_CATS:
                return self._deny(who, case, action, f"피드백 분류 오류({cat}), 허용 {sorted(FB_CATS)}")
            c["feedback"].append((cat, reason))
            self.audit.add(event="FEEDBACK", who=who, role=role, case=case, cat=cat, reason=reason, evidence=evidence); return True, "지휘관 피드백 기록"
        if action == "merge":
            if c["state"] != "군수관검토":
                return self._deny(who, case, action, f"병합은 군수관 검토 단계에서만 가능(현재 {c['state']})")
            lo, hi = min(c["a"], c["b"]), max(c["a"], c["b"])
            if value is None or not (lo - 0.2 * abs(lo) <= value <= hi + 0.2 * abs(hi)):
                return self._deny(who, case, action, f"병합 값 범위 초과(허용 {lo - 0.2 * abs(lo):.0f}~{hi + 0.2 * abs(hi):.0f}, 입력 {value})")
            before, c["merged"] = c["merged"], value
            self.audit.add(event="MERGE", who=who, role=role, case=case, before=before, after=value, a=c["a"], b=c["b"], reason=reason, evidence=evidence); return True, f"병합 값 {before} -> {value}"
        if action == "reject":
            if c["state"] in ("제안", "반려"):
                return self._deny(who, case, action, f"반려할 수 없는 단계({c['state']})")
            c["state"] = "종결(반려)" if c["version"] > MAX_REDRAFT else "반려"
            self.audit.add(event="REJECTED", who=who, role=role, case=case, version=c["version"], reason=reason, evidence=evidence); return True, c["state"]
        if action == "redraft":
            if c["state"] != "반려":
                return self._deny(who, case, action, f"재기안은 반려된 건만 가능(현재 {c['state']})")
            if who != c["submitter"]:
                return self._deny(who, case, action, "재기안은 최초 상신자만 가능")
            c["state"], c["version"] = "상신", c["version"] + 1
            self.audit.add(event="REDRAFT", who=who, role=role, case=case, version=c["version"], reason=reason, evidence=evidence); return True, "재기안 상신"
        if c["state"] == "반려":
            return self._deny(who, case, action, "반려된 건은 재기안 후에만 진행 가능")
        need = {"제안": "submit", "상신": "review", "군수관검토": "approve" if c["high"] else "confirm"}[c["state"]]
        if action != need:
            return self._deny(who, case, action, f"단계 위반(현재 {c['state']}{'·영향 큼' if c['high'] else ''}, 필요 행위 {need})")
        if action == "confirm" and c["merged"] is None or action == "approve" and c["merged"] is None:
            return self._deny(who, case, action, "군수관 병합 값 없음(자동 확정 금지)")
        if action == "submit":
            c["submitter"] = who
        c["state"] = {"submit": "상신", "review": "군수관검토", "confirm": "확정", "approve": "확정"}[action]
        self.audit.add(event=action.upper(), who=who, role=role, case=case, version=c["version"], to=c["state"], merged=c["merged"], reason=reason, evidence=evidence); return True, c["state"]


# ---- 충돌 건 12개: 같은 품목의 재고 실사 수량·최소 보유 기준을 두 보급소가 서로 다르게 수정 ----
rng = np.random.default_rng(5)
cands = []
for k in range(12):
    it = ITEMS[k]; field = "실사 수량" if k % 3 else "최소 보유 기준"
    base = 500 if field == "실사 수량" else it["min_level"]
    a = int(base * rng.uniform(0.9, 1.1)); b = int(a * rng.choice([1.05, 1.1, 1.4, 1.8, 0.5]))
    cover = rng.uniform(1.5, 14)                                         # 소진 예상 일수 (납기 대비 부족하면 공급망 재난 징후)
    risk = 100 * (0.5 * min(abs(a - b) / max(a, b) / 0.5, 1) + 0.5 * max(0, 1 - cover / it["lead_days"]))
    cands.append({"id": f"C{k + 1:02d}", "item": it["item"], "field": field, "a": a, "b": b, "cover": cover, "lead": it["lead_days"], "risk": risk})
cd = pd.DataFrame(cands)
print(f"[충돌 건] {len(cd)}건, 영향 큼(차이 {IMPACT_CUT:.0%} 이상) {sum(abs(r.a - r.b) / max(r.a, r.b) >= IMPACT_CUT for r in cd.itertuples())}건, 위험도 평균 {cd['risk'].mean():.0f}")

# ---- 큐 시뮬레이션: 군수관 1명(건당 2시간), 건이 1시간 간격으로 도착 ----
def run_queue(policy, risks, service=2.0, arr_gap=1.0):
    # [TODO : 학생 실습 7] 동기화 충돌 발생 시 군수관 수동 병합(Conflict Resolution) HITL 워크플로우 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rs = list(rng.permutation(cd["risk"].to_numpy()))
wf_, wp_ = run_queue("fifo", rs), run_queue("prio", rs)
hi = [i for i, r in enumerate(rs) if r >= 50]
print(f"\n[검토 큐 대기시간(h)] 건 {len(rs)}개, 위험 50 이상 {len(hi)}건, 허용 {SLA_H:.0f}h")
for nm, w in (("선입선출", wf_), ("위험도 우선(노화 가산)", wp_)):
    print(f"  {nm:14s}: 전체 평균 {np.mean(list(w.values())):.1f}h, 고위험 평균 {np.mean([w[i] for i in hi]):.1f}h, 최대 {max(w.values()):.1f}h, 허용 초과 {sum(v > SLA_H for v in w.values())}건")

# ---- 워크플로우 모의 ----
wf = Workflow(); res = []
for k, r in cd.iterrows():
    case = r["id"]
    wf.act("sys", case, "propose", reason=f"{r['item']} {r['field']} 충돌: A {r['a']} / B {r['b']}", cf=(int(r["a"]), int(r["b"])))
    wf.act("kim", case, "submit", "수동 병합 검토 요청", f"소진 예상 {r['cover']:.1f}일, 납기 {r['lead']}일")
    for _ in range(12):
        c = wf.cases[case]; s = c["state"]
        if s in ("확정", "종결(반려)"):
            break
        if s == "상신":
            if k in (2, 11) and c["version"] <= (1 if k == 2 else 9):           # 2번 건은 1회 반려, 11번 건은 보완 없이 계속 반려(재기안 한도 시연)
                wf.act("lee", case, "reject", "근거 사진·실사표 누락, 보완 요청", f"충돌 두 값 {r['a']}/{r['b']}")
            else:
                wf.act("lee", case, "review", "두 기록과 실사 근거 확인", f"소진 예상 {r['cover']:.1f}일")
        elif s == "군수관검토":
            if c["merged"] is None:
                val = int((c["a"] + c["b"]) / 2) if r["field"] == "실사 수량" else max(c["a"], c["b"])           # 실사는 평균 후보, 기준은 높은 쪽(안전 우선)
                wf.act("lee", case, "merge", "수동 병합: 현장 실사·납기 고려해 값 결정", f"A {c['a']} / B {c['b']}", value=val)
            if c["high"]:
                wf.act("park", case, "feedback", "영향이 큰 건이므로 선별 확인 후 결정", "우선순위", cat="우선순위")
                wf.act("park", case, "approve", "승인: 병합 값 확정", f"차이 {abs(c['a'] - c['b']) / max(c['a'], c['b']):.0%}, 군수관 병합 검토 완료")
            else:
                wf.act("lee", case, "confirm", "확정: 병합 값 반영", "차이가 작아 군수관 확정 범위")
        elif s == "반려":
            wf.act("kim", case, "redraft", "반려 사유 반영해 재기안", f"근거 보완 {c['version']}차")
    c = wf.cases[case]; res.append((case, r["field"], c["a"], c["b"], "-" if c["merged"] is None else str(c["merged"]), "지휘관" if c["high"] else "군수관", c["state"], c["version"]))
print("\n[처리 결과] (모의 결재자 기준, 실제 판단은 사람이 한다)")
print(pd.DataFrame(res, columns=["건", "항목", "A값", "B값", "병합값", "확정권자", "상태", "버전"]).head(12).to_string(index=False))
print(f"군수관 수동 병합 {sum(e['event'] == 'MERGE' for e in wf.audit.log)}건(전후 값·사유 기록), 반려 {sum(e['event'] == 'REJECTED' for e in wf.audit.log)}건(사유 필수), 지휘관 피드백 {sum(e['event'] == 'FEEDBACK' for e in wf.audit.log)}건")

# ---- 권한 우회·절차 위반 시도 ----
def mk(cid, stage="submit", ab=(100, 160)):
    wf.act("sys", cid, "propose", reason="시험", cf=ab)
    if stage in ("submit", "review"):
        wf.act("kim", cid, "submit", "시험 상신", "근거")
    if stage == "review":
        wf.act("lee", cid, "review", "검토", "이상 없음")
    return cid
done_case = next(c for c, v in wf.cases.items() if v["state"] == "확정")
att = [("시스템 자동 병합", lambda: wf.act("sys", mk("T01", "review"), "merge", "자동", "자동", value=130)), ("보급담당이 병합 시도", lambda: wf.act("kim", mk("T02", "review"), "merge", "임의", "근거", value=130)),
       ("병합 없이 확정", lambda: wf.act("lee", mk("T03", "review", (100, 105)), "confirm", "바로 확정", "근거")), ("미등록 사용자", lambda: wf.act("guest", mk("T04"), "review", "열람", "근거")),
       ("사유 없는 검토", lambda: wf.act("lee", mk("T05"), "review", "", "")), ("감사 로그 수정", lambda: wf.act("kim", mk("T06"), "edit_log", "정리", "근거")),
       ("감사관이 확정 시도", lambda: wf.act("jung", mk("T07", "review"), "confirm", "확정", "근거")), ("존재하지 않는 건", lambda: wf.act("park", "X99", "approve", "승인", "근거")),
       ("종결된 건 재승인", lambda: wf.act("park", done_case, "approve", "재승인", "근거")), ("병합 값 범위 초과(100/160 -> 500)", lambda: wf.act("lee", mk("T08", "review"), "merge", "대폭 변경", "근거", value=500)),
       ("영향 큰 건을 군수관이 확정", lambda: (wf.act("lee", "T09", "merge", "병합", "근거", value=130) if mk("T09", "review") else None, wf.act("lee", "T09", "confirm", "확정", "근거"))[-1]),
       ("지휘관이 검토 건너뛰기", lambda: wf.act("park", mk("T10"), "approve", "바로 승인", "근거")), ("피드백 분류 오류", lambda: wf.act("park", mk("T11", "review"), "feedback", "의견", "근거", cat="잡담")),
       ("상신 전 병합", lambda: wf.act("lee", mk("T12", "submit"), "merge", "병합", "근거", value=130))]
print("\n[권한 우회·절차 위반 시도]")
blocked = 0
for n, fn in att:
    ok, why = fn(); blocked += (not ok); print(f"  {n:26s}-> {'차단' if not ok else '통과(결함)'}: {why}")
print(f"차단 {blocked}/{len(att)} ({blocked / len(att):.0%})")

# ---- 로깅 성공률과 이력 무결성 ----
n_log = len(wf.audit.log); bad = wf.audit.verify()
print(f"\n처리 요청 {wf.calls}건 -> 감사 로그 {n_log}건(확정·반려·병합·거부 모두 기록), 로깅 성공률 {n_log / wf.calls:.0%} (목표 100%: {'달성' if n_log == wf.calls else '미달'})")
print(f"해시 체인 검증: {'정상' if bad is None else f'seq {bad} 불일치'} -> 이력 무결성 {'100%' if bad is None else '훼손'}")
saved = json.dumps(wf.audit.log[12], ensure_ascii=False); wf.audit.log[12]["reason"] = "사후 수정"; v = wf.audit.verify()
print(f"12번째 기록 임의 수정 후 검증: {'위변조 탐지(seq ' + str(v) + ')' if v is not None else '탐지 실패'}"); wf.audit.log[12] = json.loads(saved)
print("※ 충돌 병합은 군수관이 하고, 영향이 큰 건은 지휘관이 승인한다. 시스템은 두 값과 근거만 제시하며 자동 단정하지 않는다. 결재선은 소속 기관 규정을 따른다.")

plt.figure(figsize=(6, 3.5)); plt.bar(["FIFO", "priority"], [np.mean([wf_[i] for i in hi]), np.mean([wp_[i] for i in hi])], color=["tab:gray", "tab:blue"])
plt.ylabel("mean wait of high-risk conflicts (h)"); plt.title("Review queue wait: FIFO vs risk priority"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[충돌 건] 12건, 영향 큼(차이 25% 이상) 5건, 위험도 평균 37

[검토 큐 대기시간(h)] 건 12개, 위험 50 이상 6건, 허용 4h
  선입선출          : 전체 평균 5.5h, 고위험 평균 5.5h, 최대 11.0h, 허용 초과 7건
  위험도 우선(노화 가산) : 전체 평균 5.5h, 고위험 평균 0.8h, 최대 13.0h, 허용 초과 5건

[처리 결과] (모의 결재자 기준, 실제 판단은 사람이 한다)
  건       항목  A값  B값 병합값 확정권자     상태  버전
C01 최소 보유 기준  31  32  32  군수관     확정   1
C02    실사 수량 478 239 358  지휘관     확정   1
C03    실사 수량 488 683 585  지휘관     확정   2
C04 최소 보유 기준  54  75  75  지휘관     확정   1
C05    실사 수량 515 927 721  지휘관     확정   1
C06    실사 수량 547 601 574  군수관     확정   1
C07 최소 보유 기준  42  44  44  군수관     확정   1
C08    실사 수량 517 568 542  군수관     확정   1
C09    실사 수량 505 252 378  지휘관     확정   1
C10 최소 보유 기준  63  69  69  군수관     확정   1
C11    실사 수량 537 590 563  군수관     확정   1
C12    실사 수량 537 590   -  군수관 종결(반려)   3
군수관 수동 병합 11건(전후 값·사유 기록), 반려 4건(사유 필수), 지휘관 피드백 5건

[권한 우회·절차 위반 시도]
  시스템 자동 병합                 -> 차단: 권한 없음(시스템 역할에 merge 권한 없음)
  보급담당이 병합 시도               -> 차단: 권한 없음(보급담당 역할에 merge 권한 없음)
  병합 없이 확정                  -> 차단: 군수관 병합 값 없음(자동 확정 금지)
  미등록 사용자                   -> 차단: 미등록 사용자
  사유 없는 검토                  -> 차단: 사유·근거 입력 필수(반려·병합 사유 포함)
  감사 로그 수정                  -> 차단: 감사 로그 삭제·수정 권한은 누구에게도 없음
  감사관이 확정 시도                -> 차단: 권한 없음(감사관 역할에 confirm 권한 없음)
  존재하지 않는 건                 -> 차단: 존재하지 않는 건
  종결된 건 재승인                 -> 차단: 이미 종결된 건(확정)
  병합 값 범위 초과(100/160 -> 500)-> 차단: 병합 값 범위 초과(허용 80~192, 입력 500)
  영향 큰 건을 군수관이 확정           -> 차단: 단계 위반(현재 군수관검토·영향 큼, 필요 행위 approve)
  지휘관이 검토 건너뛰기              -> 차단: 단계 위반(현재 상신·영향 큼, 필요 행위 review)
  피드백 분류 오류                 -> 차단: 피드백 분류 오류(잡담), 허용 ['기타', '우선순위', '자원', '절차']
  상신 전 병합                   -> 차단: 병합은 군수관 검토 단계에서만 가능(현재 상신)
차단 14/14 (100%)

처리 요청 115건 -> 감사 로그 115건(확정·반려·병합·거부 모두 기록), 로깅 성공률 100% (목표 100%: 달성)
... (이하 생략)
```
</details>
